# 08. Results: exposure, forecasts, hedging value, costs, and where it comes from

**The question.** Over the final test period, does refinancing information improve the hedge beyond the updating benchmark, how much of any improvement survives trading costs, and in which environments does it appear?

**Why a portfolio manager cares.** This is the decision evidence: risk reduced, risk left over, turnover, and cost, with uncertainty, for the rules a mortgage desk would actually choose between.

**Learning objectives**

1. Measure exposure variation on the full sample (Q1).
2. Compare walk-forward forecasts against persistence, AR(1), and updating (Q2).
3. Report the main hedging table with bootstrap intervals and the four declared comparisons (Q3).
4. Apply the declared cost verdict across the cost grid, rebalancing rules, delays, regimes, and episodes (Q4).
5. Run the declared robustness checks and classify the outcome by the pre-declared rule.

| | |
|---|---|
| Inputs | daily and monthly panels, forecasts, `results/backtest/*`, raw tables (robustness reruns), freeze record |
| Outputs | `results/evaluation/*.csv|json`, figures `08_*` |
| Run after | 07 |
| Approximate runtime | 1 to 3 minutes |

In [ ]:
# Setup: find the project root, load the configuration, and pick the run mode.
# The mode comes from MBSHEDGE_MODE (set by run_project.py) or config/project.yaml.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "config" / "project.yaml").exists())
sys.path.insert(0, str(ROOT / "src"))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mbshedge.config import setup, experiment
from mbshedge import plotting as pt

cfg, paths, MODE = setup()
EXP = experiment(cfg, MODE)
pt.setup_style()
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [ ]:
from mbshedge import pipeline as pl
from mbshedge import evaluation as ev
from mbshedge import exposure as ex
from mbshedge import freeze as fz
from mbshedge.sources import RawStore, load_raw_tables
rec = fz.require_freeze(cfg, paths)
fz.log_test_evaluation(paths, rec, "08", "test-period evaluation and robustness")
daily = pd.read_parquet(paths.processed / "daily_panel.parquet")
monthly = pd.read_parquet(paths.processed / "monthly_panel.parquet")
fc_test = pl.load_forecasts(paths.results / "forecasts" / "forecasts_test.csv")
choices = rec["development_choices"]; halflife = int(choices["ewma_halflife_days"]); static_d = float(choices["static_duration_years"])
win = pl.windows(daily, EXP)
H, EV = EXP["hedging"], EXP["evaluation"]; N = EXP["notional_usd"]
PAIRS = [tuple(p) for p in EXP["models"]["primary_comparisons"]]
edir = paths.results / "evaluation"; edir.mkdir(parents=True, exist_ok=True)
base = pl.load_runs(paths.results / "backtest", "test_base")
weeks = base["UNH"]["weekly"].index
print(f"Test period: {weeks.min().date()} to {weeks.max().date()} ({len(weeks)} weeks). Frozen half-life {halflife} days, static duration {static_d:.2f} years.")

## Q1. Does measured exposure change meaningfully? (full sample, measurement)

This is a measurement question, so it uses every month. The p-value comes from the constant-duration residual bootstrap validated in notebook 04, and the size of the variation is the robust between-month standard deviation of true duration.

In [ ]:
e_cfg = EXP["exposure"]
q_full = ex.null_bootstrap_variation(daily, str(win["first_month"]), str(win["test_end"]), reps=e_cfg["q_bootstrap_reps"], seed=EXP["seed"] + 1,
                                     nw_lags=e_cfg["newey_west_lags"], min_days=e_cfg["min_days_per_month"], block_days=e_cfg["q_bootstrap_block_days"])
q_test = ex.null_bootstrap_variation(daily, str(win["test_start"]), str(win["test_end"]), reps=e_cfg["q_bootstrap_reps"], seed=EXP["seed"] + 2,
                                     nw_lags=e_cfg["newey_west_lags"], min_days=e_cfg["min_days_per_month"], block_days=e_cfg["q_bootstrap_block_days"])
vm = monthly[monthly["valid"]]
q1 = {"full": q_full, "test": q_test, "p05_D": float(vm["D"].quantile(0.05)), "p95_D": float(vm["D"].quantile(0.95)),
      "min_D": float(vm["D"].min()), "max_D": float(vm["D"].max()), "median_se": float(vm["se"].median())}
(edir / "q1_variation.json").write_text(json.dumps(q1, indent=2))
pd.DataFrame({"full sample": q_full, "test period": q_test}).loc[["months", "pooled_duration", "sd_observed", "sd_noise_under_null", "tau_robust", "Q", "df", "p_bootstrap", "p_chi2_naive"]]

In [ ]:
ish = cfg["data"]["ishares_mbb_reference"]
last = monthly[monthly["valid"]].iloc[-1]
fig, ax = plt.subplots(figsize=(11, 4))
vm2 = monthly[monthly["valid"]]
ax.errorbar(vm2["decision_date"], vm2["D"], yerr=vm2["se"], fmt="o", ms=2.5, color="#2a78d6", ecolor="#cde2fb", lw=0.7, label="realized monthly duration +/- 1 se")
ax.plot(monthly["decision_date"], monthly[f"U_{halflife}"], color="#eb6834", lw=1.5, label=f"updating (EWMA, {halflife}-day half-life)")
ax.scatter([pd.Timestamp(ish["as_of"])], [ish["effective_duration_years"]], marker="D", color=pt.INK, zorder=4, label=f"iShares effective duration ({ish['as_of']})")
pt.mark_test_start(ax, EXP["sample"]["test_start"] + "-01")
pt.shade_episodes(ax, EXP["episodes"], end_default=daily.index.max())
ax.set_ylabel("years"); ax.set_title("Empirical duration of MBB, monthly"); ax.legend(loc="upper left", fontsize=8.5)
pt.save_figure(fig, "08_realized_duration_full", paths, data=monthly[["decision_date", "D", "se", f"U_{halflife}"]], mode=MODE)
plt.show()
print(f"Latest valid month {last.name}: realized empirical duration {last['D']:.2f} +/- {last['se']:.2f} years, updating estimate "
      f"{last[f'U_{halflife}']:.2f} years. iShares reported an option-adjusted effective duration of {ish['effective_duration_years']} years "
      f"on {ish['as_of']}. The two measure different things (empirical co-movement versus a model with fixed spreads), so this is a sanity check, not a test.")

## Q2. Does refinancing information forecast next month's exposure?

All models are compared on the same test months, against the realized duration of the target month. The realized target is itself noisy, which adds the same expected amount to every model's mean squared error, so differences between models remain fair. The Clark–West test applies to the nested pair H vs U (H equals U when the refinancing slope is zero). Diebold–Mariano compares the non-nested E and U.

In [ ]:
models_eval = ["S", "U", "P", "AR", "E", "H"]
ftab = ev.forecast_table(fc_test, models_eval, reference="U")
wide = fc_test.pivot_table(index="target_month", columns="model", values="forecast")
real = fc_test.drop_duplicates("target_month").set_index("target_month")[["realized", "realized_valid"]]
dfc = wide.join(real); dfc = dfc[dfc["realized_valid"].astype(bool)].dropna(subset=models_eval)
lags_m = EV["hac_lags_monthly"]
cw = ev.clark_west(dfc["realized"], dfc["U"], dfc["H"], lags_m)
dm_eu = ev.diebold_mariano(dfc["realized"], dfc["U"], dfc["E"], lags_m)
dm_hp = ev.diebold_mariano(dfc["realized"], dfc["P"], dfc["H"], lags_m)
ftests = {"clark_west_H_vs_U": cw, "diebold_mariano_U_vs_E": dm_eu, "diebold_mariano_P_vs_H": dm_hp, "n_months": int(len(dfc))}
ftab.to_csv(edir / "forecast_table_test.csv", index=False); (edir / "forecast_tests_test.json").write_text(json.dumps(ftests, indent=2))
display(ftab.drop(columns=["noise_adjusted_mse"]).round(3))
print(f"Clark-West (does H improve on U?): statistic {cw['cw_stat']:.2f}, one-sided p = {cw['p_one_sided']:.3f} over {cw['n']} months")
print(f"Diebold-Mariano E vs U: mean loss difference {dm_eu['mean_loss_diff']:+.3f} (positive favors E), two-sided p = {dm_eu['p_two_sided']:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6.5), sharex=True)
t_idx = dfc.index.to_timestamp(how="end")
axes[0].scatter(t_idx, dfc["realized"], s=10, color="#9ec5f4", label="realized duration (target month)")
for s in ["S", "U", "E", "H"]:
    st = pt.style(s); axes[0].plot(t_idx, dfc[s], color=st["color"], ls=st["linestyle"], lw=1.5, label=st["label"])
axes[0].set_ylabel("years"); axes[0].set_title("Forecasts of next month's duration (test period)"); axes[0].legend(ncol=3, fontsize=8)
for s in ["H", "E", "P", "AR"]:
    st = pt.style(s)
    cum = ((dfc["realized"] - dfc["U"]) ** 2 - (dfc["realized"] - dfc[s]) ** 2).cumsum()
    axes[1].plot(t_idx, cum, color=st["color"], ls=st["linestyle"], lw=1.5, label=st["label"])
axes[1].axhline(0, color=pt.INK_2, lw=0.8)
axes[1].set_ylabel("years^2"); axes[1].set_title("Cumulative squared-error advantage over the updating forecast (rising = better than U)")
axes[1].legend(ncol=2, fontsize=8)
for a in axes: pt.shade_episodes(a, [e for e in EXP["episodes"] if e["period"] == "test"], end_default=daily.index.max())
pt.save_figure(fig, "08_forecast_comparison", paths, data=dfc, mode=MODE)
plt.show()

## Q3. Does it lower out-of-sample hedged risk?

The main table uses weekly hedged excess P&L per dollar of MBS over the test period, at the base settings (monthly rebalancing, one-day execution delay, 2 bp one-way cost). Volatility is annualized. Variance reductions compare gross P&L. The ex-post diagnostic (X) uses next month's realized duration, which no one could know in advance. It is shown for context only and is not an upper bound.

The paired bootstrap resamples blocks of weeks and recomputes every comparison on the same resampled weeks. Statistical support means the 95% interval lies above zero and, for the four declared comparisons, the Holm-adjusted one-sided p-value for "no variance reduction" is below 0.025. The p-value is read from the same bootstrap distribution as the interval, so the two never disagree.

In [ ]:
weekly = {s: r["weekly"] for s, r in base.items()}
infos = {s: r["info"] for s, r in base.items()}
summary = ev.strategy_summary(weekly, infos, N, lags=EV["hac_lags_weekly"])
W = pd.DataFrame({s: w["gross_pnl"] for s, w in weekly.items()})
for comp in ["S", "U"]:
    summary[f"var_reduction_vs_{comp}"] = [ev.variance_reduction(W[s], W[comp]) for s in summary.index]
summary.to_csv(edir / "main_table_test.csv")
cols = ["vol_gross_ann_pct", "var_reduction_vs_unhedged", "var_reduction_vs_S", "var_reduction_vs_U", "residual_duration_years",
        "residual_duration_se", "es95_weekly_net_bp", "max_drawdown_net_pct", "mean_net_ann_pct", "turnover_per_year", "cost_drag_bp_per_year", "position_limit_hits"]
summary[cols].round(3)

In [ ]:
boot = ev.paired_bootstrap(W, PAIRS + [("S", "UNH"), ("E", "S")], EV["bootstrap_reps"], EV["bootstrap_block_weeks"], EXP["seed"], EV["confidence"])
boot = ev.add_holm(boot, PAIRS)   # Holm over the four declared comparisons only
boot.to_csv(edir / "bootstrap_test.csv", index=False)
boot[["challenger", "comparator", "primary", "variance_reduction", "vr_ci_low", "vr_ci_high", "vol_reduction_ann", "p_one_sided", "p_holm"]].round(4)

In [ ]:
verdicts = pd.DataFrame([ev.cost_verdict(summary, boot, a, b, EV["hurdle_information_ratio"], H["base_cost_bp"],
                                         EV["practical_variance_reduction_vs_U"] if b == "U" else EV["practical_variance_reduction_vs_S"]) for a, b in PAIRS])
verdicts.to_csv(edir / "verdicts_test.csv", index=False)
vx = summary.loc["X", "vol_gross_ann_pct"] ** 2; vs = summary.loc["S", "vol_gross_ann_pct"] ** 2
share_rows = []
for s in ["U", "E", "H"]:
    denom = vs - vx
    share = (vs - summary.loc[s, "vol_gross_ann_pct"] ** 2) / denom if denom > 0.05 * vs else np.nan
    share_rows.append({"strategy": s, "share_of_static_to_expost_gap": share})
pd.DataFrame(share_rows).to_csv(edir / "expost_gap_share.csv", index=False)
verdicts[["challenger", "comparator", "variance_reduction", "vr_ci_low", "meets_practical_target", "value_of_risk_reduction_bp_per_year",
          "extra_cost_bp_per_year", "net_gain_bp_per_year", "breakeven_cost_bp_per_trade", "net_gain_positive_when", "statistical_support", "verdict"]].round(3)

In [ ]:
for _, v in verdicts.iterrows():
    a, b = v["challenger"], v["comparator"]
    print(f"{pt.STRATEGY_STYLE[a]['label']} vs {pt.STRATEGY_STYLE[b]['label']}: variance reduction {v['variance_reduction']:+.1%} "
          f"(95% interval {v['vr_ci_low']:+.1%} to {float(boot.loc[(boot.challenger==a)&(boot.comparator==b),'vr_ci_high'].iloc[0]):+.1%}), "
          f"risk value {v['value_of_risk_reduction_bp_per_year']:+.2f} bp/yr against extra cost {v['extra_cost_bp_per_year']:+.2f} bp/yr "
          f"-> {v['verdict']}. " + ev.breakeven_text(v))
print(f"\nIn dollars per ${N/1e6:.0f}mm: annualized hedged volatility is ${summary.loc['S','vol_gross_usd_mm']:.2f}mm static, "
      f"${summary.loc['U','vol_gross_usd_mm']:.2f}mm updating, ${summary.loc['H','vol_gross_usd_mm']:.2f}mm hybrid; "
      f"annual trading cost ${summary.loc['S','cost_usd_per_year']:,.0f}, ${summary.loc['U','cost_usd_per_year']:,.0f}, and ${summary.loc['H','cost_usd_per_year']:,.0f}.")

## Q4a. Costs, rebalancing rules, and execution delay

The cost verdict is recomputed at every cost level in the grid. Each rule and delay is a separate rerun of the engine (notebook 07). The frontier plots annualized hedged volatility against annual cost drag: down and to the left is better.

In [ ]:
grid = pd.read_parquet(paths.results / "backtest" / "weekly_test_grid.parquet")
rows = []
for (s, rule, delay, fin), g in grid.groupby(["strategy", "rule", "delay", "financing_bp"]):
    g = g.set_index("week")
    yrs = (g.index.max() - g.index.min()).days / 365.25
    for c in H["cost_grid_bp"]:
        rows.append({"strategy": s, "rule": rule, "delay": delay, "financing_bp": fin, "cost_bp": c,
                     "vol_gross_ann_pct": 100 * ev.ann_vol(g["gross_pnl"]),
                     "cost_bp_per_year": 1e4 * (c * g["cost_1bp"].sum() + g["financing"].sum()) / yrs,
                     "turnover_per_year": float(g["turnover_per_year"].iloc[0])})
gm = pd.DataFrame(rows); gm.to_csv(edir / "grid_metrics_test.csv", index=False)
cost_rows = []
for c in H["cost_grid_bp"]:
    for a, b in PAIRS:
        v = ev.cost_verdict(summary, boot, a, b, EV["hurdle_information_ratio"], c, 0.0)
        cost_rows.append({"cost_bp": c, "comparison": f"{a} vs {b}", "net_gain_bp_per_year": v["net_gain_bp_per_year"], "verdict": v["verdict"]})
cost_table = pd.DataFrame(cost_rows); cost_table.to_csv(edir / "cost_grid_verdicts_test.csv", index=False)
cost_table.pivot(index="comparison", columns="cost_bp", values="verdict")

In [ ]:
base_rule, base_delay = H["base_rule"], H["base_delay_days"]
fr = gm[(gm["delay"] == base_delay) & (gm["financing_bp"] == 0) & (gm["cost_bp"] == H["base_cost_bp"]) & gm["strategy"].isin(["S", "U", "E", "H"])]
fig, ax = plt.subplots(figsize=(8, 5))
markers = {"monthly": "o", "quarterly": "s", "band_0.10": "^", "band_0.25": "v", "band_0.50": "D"}
for s in ["S", "U", "E", "H"]:
    d = fr[fr["strategy"] == s]
    st = pt.style(s)
    for _, r in d.iterrows():
        ax.scatter(r["cost_bp_per_year"], r["vol_gross_ann_pct"], color=st["color"], marker=markers[r["rule"]], s=46, zorder=3)
    ax.scatter([], [], color=st["color"], marker="o", label=st["label"])
for rule, mk in markers.items():
    ax.scatter([], [], color=pt.INK_2, marker=mk, label=rule)
ax.set_xlabel(f"annual cost drag at {H['base_cost_bp']} bp per trade (bp per year)"); ax.set_ylabel("annualized hedged volatility (%)")
ax.set_title("Risk against cost by rebalancing rule (test period)"); ax.legend(fontsize=8, ncol=2)
pt.save_figure(fig, "08_risk_cost_frontier", paths, data=fr, mode=MODE)
plt.show()
dl = gm[(gm["rule"] == base_rule) & (gm["financing_bp"] == 0) & (gm["cost_bp"] == H["base_cost_bp"])].pivot(index="strategy", columns="delay", values="vol_gross_ann_pct")
print("Annualized hedged volatility (%) by execution delay (trading days):"); display(dl.round(3))

## Q4b. Regimes and episodes: where does any difference come from?

Weeks are classified by information available at the start of each week: the refinancing incentive (locked in below −0.5 pp, refinanceable above +0.5 pp, near the money in between) and rate volatility (63-day realized volatility of 10-year yield changes above or below its expanding median). Episodes are the windows declared in the configuration before the test. These slices describe where results come from. They are not separate hypothesis tests.

In [ ]:
prev = daily[["incentive", "dy10"]].copy()
prev["vol63"] = prev["dy10"].rolling(63).std()
prev["vol_med"] = prev["vol63"].expanding(min_periods=126).median()
start_state = prev.reindex(pd.date_range(prev.index.min(), prev.index.max())).ffill()
wk_state = start_state.reindex(weeks - pd.Timedelta(days=7))
wk_state.index = weeks
R = EV["regimes"]
moneyness = np.where(wk_state["incentive"] <= R["locked_in_below_pp"], "locked in",
                     np.where(wk_state["incentive"] >= R["refinanceable_above_pp"], "refinanceable", "near the money"))
volreg = np.where(wk_state["vol63"] > wk_state["vol_med"], "high rate volatility", "low rate volatility")
reg_rows = []
for label, lab_arr in [("moneyness", moneyness), ("volatility", volreg)]:
    for val in sorted(set(lab_arr)):
        sel = W[lab_arr == val]
        if len(sel) < 26:
            continue
        b = ev.paired_bootstrap(sel, [("H", "U"), ("U", "S"), ("E", "U")], min(1000, EV["bootstrap_reps"]), EV["bootstrap_block_weeks"], EXP["seed"] + 7)
        for _, r in b.iterrows():
            reg_rows.append({"dimension": label, "regime": val, "weeks": len(sel), "comparison": f"{r['challenger']} vs {r['comparator']}",
                             "variance_reduction": r["variance_reduction"], "ci_low": r["vr_ci_low"], "ci_high": r["vr_ci_high"]})
regimes = pd.DataFrame(reg_rows); regimes.to_csv(edir / "regimes_test.csv", index=False)
regimes.round(3)

In [ ]:
ep_rows = []
ss = (W ** 2)
tot_hu = float(ss["U"].sum() - ss["H"].sum())
for e in EXP["episodes"]:
    if e["period"] != "test":
        continue
    end = pd.Timestamp(e["end"]) if e["end"] else weeks.max()
    sel = (weeks >= pd.Timestamp(e["start"])) & (weeks <= end + pd.Timedelta(days=6))
    if sel.sum() < 4:
        continue
    Ws = W[sel]
    out = W[~sel]
    ep_rows.append({"episode": e["name"], "weeks": int(sel.sum()),
                    "vol_S": 100 * ev.ann_vol(Ws["S"]), "vol_U": 100 * ev.ann_vol(Ws["U"]), "vol_H": 100 * ev.ann_vol(Ws["H"]),
                    "vr_H_vs_U": ev.variance_reduction(Ws["H"], Ws["U"]), "vr_U_vs_S": ev.variance_reduction(Ws["U"], Ws["S"]),
                    "share_of_total_HU_squared_pnl_reduction": float((ss["U"][sel].sum() - ss["H"][sel].sum()) / tot_hu) if abs(tot_hu) > 0 else np.nan,
                    "vr_H_vs_U_leaving_out": ev.variance_reduction(out["H"], out["U"]),
                    "cum_pnl_S_bp": 1e4 * Ws["S"].sum(), "cum_pnl_U_bp": 1e4 * Ws["U"].sum(), "cum_pnl_H_bp": 1e4 * Ws["H"].sum()})
episodes = pd.DataFrame(ep_rows); episodes.to_csv(edir / "episodes_test.csv", index=False)
episodes.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
cum_hu = (ss["U"] - ss["H"]).cumsum() * 1e4
cum_us = (ss["S"] - ss["U"]).cumsum() * 1e4
ax.plot(weeks, cum_us, color=pt.STRATEGY_STYLE["U"]["color"], lw=1.6, label="Updating better than static (cumulative)")
ax.plot(weeks, cum_hu, color=pt.STRATEGY_STYLE["H"]["color"], lw=1.6, label="Hybrid better than updating (cumulative)")
ax.axhline(0, color=pt.INK_2, lw=0.8)
pt.shade_episodes(ax, [e for e in EXP["episodes"] if e["period"] == "test"], end_default=weeks.max())
ax.set_ylabel("cumulative reduction in squared weekly P&L (bp^2 x 1e-4)"); ax.set_title("When the risk reduction accrued (rising = the first rule had lower risk)")
ax.legend(fontsize=8.5)
pt.save_figure(fig, "08_when_gains_accrue", paths, data=pd.DataFrame({"cum_U_minus_H": cum_hu, "cum_S_minus_U": cum_us}), mode=MODE)
plt.show()

## Declared robustness checks

Each check changes one declared assumption and reruns everything it affects with the frozen choices, then recomputes the main comparisons with a paired bootstrap. No result here can change the frozen headline. A check that reverses a conclusion is reported as such.

In [ ]:
tables = load_raw_tables(RawStore(paths.raw), cfg)
info_state = json.loads((paths.results / "state" / "state_info.json").read_text())
frozen_proxy = info_state["proxy"]; sigma0 = float(choices["sigma_pp"])
rob_reps = min(1000, EV["bootstrap_reps"])

def run_variant(name, daily_v, monthly_v, static_v=static_d, e_mask=None, ret="r_mbb_ex", hcol="r_h10_ex", dcol="dur_h10", dycol="dy10",
                rule=H["base_rule"], delay=H["base_delay_days"], fin=H["financing_spread_bp_per_year"], exclude=None, block=EV["bootstrap_block_weeks"],
                fallback=False):
    fc_v = pl.forecasts(monthly_v, win, "test", static_v, halflife, ["S", "U", "E", "H"], e_train_mask_col=e_mask,
                        fallback_before_fit=fallback)
    res_v = pl.run_strategies(daily_v, fc_v, ["S", "U", "E", "H"], win["test"]["end_date"], delay=delay, rule=rule, cost_bp=H["base_cost_bp"],
                              financing_bp=fin, max_h=H["max_hedge_ratio"], ret_col=ret, hedge_ret_col=hcol, dur_col=dcol, dy_col=dycol)
    Wv = pd.DataFrame({s: r["weekly"]["gross_pnl"] for s, r in res_v.items()})
    if exclude:
        Wv = Wv[(Wv.index < pd.Timestamp(exclude[0])) | (Wv.index > pd.Timestamp(exclude[1]) + pd.Timedelta(days=6))]
    b = ev.add_holm(ev.paired_bootstrap(Wv, PAIRS, rob_reps, block, EXP["seed"] + 11), PAIRS)
    sm_v = ev.strategy_summary({s: r["weekly"].loc[Wv.index] for s, r in res_v.items()}, {s: r["info"] for s, r in res_v.items()}, N)
    vh = ev.cost_verdict(sm_v, b, "H", "U", EV["hurdle_information_ratio"], H["base_cost_bp"], EV["practical_variance_reduction_vs_U"])
    row = {"check": name, "weeks": len(Wv),
           "fallback_decisions": int(fc_v.loc[fc_v["model"] == "H", "fallback"].sum())}
    for _, r in b.iterrows():
        k = f"{r['challenger']}_vs_{r['comparator']}"
        row[f"vr_{k}"] = r["variance_reduction"]; row[f"ci_low_{k}"] = r["vr_ci_low"]; row[f"ci_high_{k}"] = r["vr_ci_high"]
    row["verdict_H_vs_U"] = vh["verdict"]
    return row

rob = [run_variant("base (reproduces the main table)", daily, monthly)]
rob.append(run_variant("execution at the decision close (0 days)", daily, monthly, delay=0))
rob.append(run_variant("execution 5 trading days later", daily, monthly, delay=5))
rob.append(run_variant("quarterly rebalancing", daily, monthly, rule="quarterly"))
rob.append(run_variant("0.25-year duration band", daily, monthly, rule="band_0.25"))
rob.append(run_variant(f"financing spread {H['financing_spread_sensitivity_bp']} bp/yr on the short", daily, monthly, fin=H["financing_spread_sensitivity_bp"]))
for blk in EV["bootstrap_block_robustness"]:
    rob.append(run_variant(f"bootstrap mean block {blk} weeks", daily, monthly, block=blk))
rob.append(run_variant("excluding March-April 2020 (ETF price dislocation)", daily, monthly, exclude=EV["excluded_window_robustness"]))

def state_variant(name, **kw):
    prep_v = pl.prepare(tables, cfg, EXP, **kw)
    mon_v = pl.monthly_panel(prep_v["daily"], EXP, [halflife])
    return run_variant(name, prep_v["daily"], mon_v)

rob.append(state_variant(f"NMDB lag {cfg['timing']['nmdb_lag_quarters_robustness']} quarters", lag_quarters=cfg["timing"]["nmdb_lag_quarters_robustness"]))
for scale in EXP["incentive"]["dispersion_robustness_scale"]:
    rob.append(state_variant(f"note-rate dispersion sigma x {scale}", sigma=round(sigma0 * scale, 3), proxy_params=frozen_proxy))
rob.append(state_variant("NMDB market: All Mortgages", markets=[cfg["data"]["nmdb"]["market_robustness"]]))
# No proxy anywhere: the refinancing share is blanked out before NMDB data exist, so the state model trains only
# on NMDB months and the hybrid's average share (gbar) uses only NMDB-era days. Until 24 NMDB training pairs
# exist, the economic model uses the static duration and the hybrid equals the updating hedge (counted below).
daily_nm = daily.copy(); daily_nm["g_nmdb"] = daily_nm["g"].where(daily_nm["incentive_source"] == "nmdb")
mon_nm = pl.monthly_panel(daily_nm, EXP, [halflife], g_col="g_nmdb")
rob.append(run_variant("state model and hybrid use NMDB data only (no proxy)", daily, mon_nm, fallback=True))
if "r_h5_ex" in daily and daily["r_h5_ex"].notna().sum() > 1000:
    mon5 = pl.monthly_panel(daily, EXP, [halflife], dy_col="dy5")
    from mbshedge.models import static_duration
    s5 = static_duration(mon5, EXP["sample"]["first_month"], EXP["sample"]["development_end"])
    rob.append(run_variant("5-year hedge (exposure measured on 5-year yields)", daily, mon5, static_v=s5, hcol="r_h5_ex", dcol="dur_h5", dycol="dy5"))
robust = pd.DataFrame(rob); robust.to_csv(edir / "robustness_test.csv", index=False)
short = robust[(robust["weeks"] != robust["weeks"].iloc[0]) & ~robust["check"].str.startswith("excluding")]
for _, r in short.iterrows():
    print(f"Note: '{r['check']}' compares {r['weeks']} weeks, against {robust['weeks'].iloc[0]} in the base case.")
for _, r in robust[robust["fallback_decisions"] > 0].iterrows():
    print(f"Note: in '{r['check']}' the hybrid equals the updating hedge (and the economic model the static hedge) at the first "
          f"{r['fallback_decisions']} test decisions, before 24 NMDB training pairs exist.")
robust[["check", "weeks", "fallback_decisions", "vr_H_vs_U", "ci_low_H_vs_U", "ci_high_H_vs_U", "vr_U_vs_S", "vr_E_vs_U", "verdict_H_vs_U"]].round(3)

## Negative convexity in the realized P&L (descriptive)

If the MBS is negatively convex relative to the Treasury hedge, monthly hedged P&L should bend downward in the yield change: losses after large moves in either direction. We regress each month's hedged P&L on the month's yield change and its square. A negative quadratic coefficient is consistent with negative convexity, but the regression also absorbs spread behavior that accompanies large rate moves, so it is a description, not a structural estimate. Updating hedges can remove the linear tilt. No Treasury-only hedge can remove the curvature.

In [ ]:
conv_rows = []
for s in ["S", "U", "H"]:
    L = pd.read_parquet(paths.results / "backtest" / f"ledger_test_base_{s}.parquet")
    mth = pd.DataFrame({"pnl": L["gross_pnl"], "dy": daily.loc[L.index, "dy10"]}).groupby(L.index.to_period("M")).sum()
    X = np.column_stack([np.ones(len(mth)), mth["dy"], mth["dy"] ** 2])
    fit = ex.ols_hac(mth["pnl"].to_numpy(), X, EV["hac_lags_monthly"])
    se = np.sqrt(np.diag(fit["cov"]))
    conv_rows.append({"strategy": s, "months": len(mth), "g1_linear_tilt_years": -fit["beta"][1], "g1_se": se[1],
                      "g2_curvature": fit["beta"][2], "g2_se": se[2], "g2_t": fit["beta"][2] / se[2]})
conv = pd.DataFrame(conv_rows); conv.to_csv(edir / "convexity_test.csv", index=False)
conv.round(3)

## Which pre-declared outcome applies?

The mapping below was fixed in `docs/research_design.md` and `docs/outcome_interpretations.md` before evaluation:

1. **Useful improvement**: the hybrid passes the survives-costs verdict against the updating hedge at the base cost and meets the 5% practical target.
2. **Small but statistically supported improvement**: the hybrid passes the survives-costs verdict but its variance reduction is below the 5% practical target.
3. **Advantage limited to particular environments**: neither of the above, the overall point estimate favors the hybrid, and in at least one declared regime slice with at least 52 weeks the hybrid's variance reduction is at least 5% with a bootstrap interval above zero.
4. **No demonstrated gain over the simple updating hedge**: otherwise. If the 95% interval for the variance reduction still reaches the 5% practical target, the result is **inconclusive**: the data cannot rule out a gain of the size a desk would care about. If the whole interval lies below 5%, the data rule out a gain of that size.

The rule is coded in `mbshedge.evaluation.classify_outcome`, which the experiment freeze covers.

In [ ]:
v_hu = verdicts[(verdicts.challenger == "H") & (verdicts.comparator == "U")].iloc[0]
b_hu = boot[(boot.challenger == "H") & (boot.comparator == "U")].iloc[0]
oc = ev.classify_outcome(v_hu.to_dict(), b_hu.to_dict(), regimes, EV["practical_variance_reduction_vs_U"])
res_out = {"outcome": oc["outcome"], "inconclusive": oc["inconclusive"], "rules_out_practical_gain": oc["rules_out_practical_gain"],
           "variance_reduction_H_vs_U": oc["variance_reduction"], "ci_low": oc["ci_low"], "ci_high": oc["ci_high"],
           "practical_target": oc["practical_target"], "verdict_H_vs_U": oc["verdict"], "p_holm_H_vs_U": float(v_hu["p_holm"]),
           "qualifying_slices": [{k: r[k] for k in ("dimension", "regime", "weeks", "variance_reduction", "ci_low")} for r in oc["qualifying_slices"]],
           "mode": MODE, "synthetic": MODE == "smoke"}
(edir / "outcome.json").write_text(json.dumps(res_out, indent=2, default=str))
note = ""
if oc["inconclusive"]:
    note = f" (inconclusive: the 95% interval reaches {oc['ci_high']:+.1%}, so a gain of {oc['practical_target']:.0%} or more is not ruled out)"
elif oc["rules_out_practical_gain"]:
    note = f" (the 95% interval ends at {oc['ci_high']:+.1%}, so a gain of {oc['practical_target']:.0%} or more is ruled out)"
print(f"Pre-declared outcome: {oc['outcome'].upper()}" + note)
print(json.dumps(res_out, indent=1, default=str))
if MODE == "smoke":
    print("\nSMOKE RUN: this classification describes synthetic data and must not be quoted.")